# 03 — Player Data

Explore player objects: stats dictionaries, injury status, schedules, and the stat key scheme.

In [ ]:
import sys
sys.path.insert(0, "../src")
from connection import get_league
import pandas as pd

league = get_league()
team = league.teams[0]
player = team.roster[0]
print(f"Sample player: {player.name} ({player.proTeam})")

## Player Attributes

In [ ]:
print(f"name:             {player.name}")
print(f"playerId:         {player.playerId}")
print(f"position:         {player.position}")
print(f"lineupSlot:       {player.lineupSlot}")
print(f"eligibleSlots:    {player.eligibleSlots}")
print(f"proTeam:          {player.proTeam}")
print(f"injuryStatus:     {player.injuryStatus}")
print(f"injured:          {player.injured}")
print(f"acquisitionType:  {player.acquisitionType}")
print(f"posRank:          {player.posRank}")
print(f"total_points:     {player.total_points}")
print(f"avg_points:       {player.avg_points}")
print(f"projected_total:  {player.projected_total_points}")
print(f"projected_avg:    {player.projected_avg_points}")

## Stats Dictionary — Key Scheme

The `player.stats` dict uses keys like `{year}_total`, `{year}_projected`, etc.

Under the hood, ESPN's raw API uses numeric keys like `002025` where:
- First 2 digits = scope (`00`=season total, `10`=projected, `01`=last 7 days, `02`=last 15, `03`=last 30)
- Remaining digits = year

The `espn-api` library translates these into readable keys.

In [ ]:
print(f"Stats keys for {player.name}:")
for key in sorted(player.stats.keys()):
    inner = player.stats[key]
    inner_keys = list(inner.keys()) if isinstance(inner, dict) else type(inner).__name__
    print(f"  '{key}': {inner_keys}")

In [ ]:
# Dig into one stats entry
year = league.year
total_key = f"{year}_total"
if total_key in player.stats:
    entry = player.stats[total_key]
    print(f"Stats entry '{total_key}':")
    for k, v in entry.items():
        if isinstance(v, dict):
            print(f"  {k}:")
            for sk, sv in sorted(v.items()):
                print(f"    {sk}: {sv}")
        else:
            print(f"  {k}: {v}")
else:
    print(f"No '{total_key}' key. Available: {list(player.stats.keys())}")

## Nine-Cat Averages

In [ ]:
print(f"Nine-cat averages for {player.name}:")
nine_cat = player.nine_cat_averages
if nine_cat:
    for cat, val in sorted(nine_cat.items()):
        print(f"  {cat}: {val}")
else:
    print("  (not available)")

## Player Schedule

Players have a `schedule` attribute with upcoming games by scoring period.

In [ ]:
print(f"Schedule for {player.name}:")
print(f"  Type: {type(player.schedule)}")
if player.schedule:
    if isinstance(player.schedule, dict):
        for period, info in sorted(player.schedule.items())[:10]:
            print(f"  Period {period}: {info}")
    else:
        for item in player.schedule[:10]:
            print(f"  {item}")
else:
    print("  (no schedule data)")

## Comparing Stats Across Multiple Players

In [ ]:
rows = []
for t in league.teams:
    for p in t.roster:
        total_key = f"{league.year}_total"
        avg = p.stats.get(total_key, {}).get('avg', {})
        if avg:
            rows.append({
                "Name": p.name,
                "Team": t.team_abbrev,
                "Pos": p.position,
                "PTS": avg.get('PTS', 0),
                "REB": avg.get('REB', 0),
                "AST": avg.get('AST', 0),
                "STL": avg.get('STL', 0),
                "BLK": avg.get('BLK', 0),
                "3PM": avg.get('3PM', 0),
                "TO": avg.get('TO', 0),
                "FG%": avg.get('FG%', 0),
                "FT%": avg.get('FT%', 0),
            })

df = pd.DataFrame(rows).sort_values("PTS", ascending=False)
df.head(20)

## Injury Report

In [ ]:
injured = []
for t in league.teams:
    for p in t.roster:
        if p.injuryStatus != "ACTIVE":
            injured.append({
                "Name": p.name,
                "Team": t.team_abbrev,
                "Status": p.injuryStatus,
                "Injured": p.injured,
            })

if injured:
    pd.DataFrame(injured)
else:
    print("No injured players on any roster.")

## Player Info Lookup

In [ ]:
# Look up a specific player by name
info = league.player_info(name=player.name)
print(f"Player info for {info.name}:")
print(f"  playerId: {info.playerId}")
print(f"  proTeam:  {info.proTeam}")
print(f"  Stats keys: {list(info.stats.keys())}")